In [1]:
# Explainable machine learning for household electricity demand forecasting.
#
# Dataset  : UCI Individual Household Electric Power Consumption
#            (December 2006 to November 2010, one-minute resolution)
# Target   : Global_active_power (kW)
# Pipeline : XGBoost, LightGBM, CatBoost and Random Forest base learners
#            combined by a constrained convex blend. The Holt-Winters
#            decomposition is retained as an analysis layer only.


In [2]:
# Package installation

import subprocess, sys

_PACKAGES = [
    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
    'seaborn>=0.12', 'scikit-learn>=1.3', 'statsmodels>=0.14',
    'shap>=0.42', 'lime>=0.2', 'scipy>=1.10',
    'xgboost>=1.7', 'lightgbm>=4.0', 'catboost>=1.2', 'joblib>=1.2',
    'arch>=5.3',  # circular block bootstrap for autocorrelated residuals
]
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q'] + _PACKAGES
)
print('Package installation complete.')


Package installation complete.


In [3]:
# Imports and global configuration

from __future__ import annotations
import gc, warnings, json, time
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, wilcoxon

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

from xgboost import XGBRegressor
import lightgbm
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
import shap
from lime.lime_tabular import LimeTabularExplainer
import joblib


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Output directory
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)

# Plot style
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi':      110,
    'figure.figsize':  (12, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size':       10,
})
PALETTE = {
    'primary':   '#1f4e8c',
    'secondary': '#c0392b',
    'accent':    '#27ae60',
    'neutral':   '#7f8c8d',
}

# Figure conventions
plt.rcParams.update({
    'font.size':       11,
    'axes.titlesize':  12,
    'axes.labelsize':  11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'savefig.dpi':     400,
    'pdf.fonttype':    42, 
    'ps.fonttype':     42,
})

def savefig_pub(stem, fig=None, **kwargs):
    """Save a figure as a 400-dpi raster and a vector copy."""
    fig = fig if fig is not None else plt.gcf()
    kwargs.setdefault('bbox_inches', 'tight')
    fig.savefig(OUTPUT_DIR / f'{stem}.png', dpi=300, **kwargs)
    fig.savefig(OUTPUT_DIR / f'{stem}.pdf', **kwargs)

# Descriptive feature labels.


_RAW_VAR_LABELS = {
    'Global_active_power':   'Active power (kW)',
    'Global_reactive_power': 'Reactive power (kVAr)',
    'Voltage':               'Voltage (V)',
    'Global_intensity':      'Current intensity (A)',
    'Sub_metering_1':        'Kitchen sub-meter (Wh)',
    'Sub_metering_2':        'Laundry sub-meter (Wh)',
    'Sub_metering_3':        'Water-heater sub-meter (Wh)',
}

def _dur_label(m):
    """Readable duration for a horizon of m minutes."""
    m = int(m)
    if m % 1440 == 0:
        d = m // 1440
        return '1 day' if d == 1 else f'{d} days'
    if m % 60 == 0:
        h = m // 60
        return '1 hour' if h == 1 else f'{h} hours'
    return '1 minute' if m == 1 else f'{m} minutes'

def pub_label(feat):
    """Descriptive label for an engineered-feature name."""
    import re as _re
    f = str(feat)
    _fixed = {
        'diff_lag1_lag2':        'Load change (1 minute)',
        'diff_lag1_lag60':       'Load change (1 hour)',
        'gap_vs_yesterday':      'Change from previous day',
        'gap_vs_last_week':      'Change from previous week',
        'gap_vs_daily_mean':     'Deviation from daily mean',
        'gap_vs_weekly_mean':    'Deviation from weekly mean',
        'interaction_hour_lag1': 'Recent load by hour of day',
        'hour_sin':   'Hour of day (sine)',
        'hour_cos':   'Hour of day (cosine)',
        'day_sin':    'Day of week (sine)',
        'day_cos':    'Day of week (cosine)',
        'month_sin':  'Month of year (sine)',
        'month_cos':  'Month of year (cosine)',
        'mon_sin':    'Month of year (sine)',
        'mon_cos':    'Month of year (cosine)',
        'is_weekend': 'Weekend indicator',
        'hw_trend':        'Holt-Winters trend',
        'hw_seasonal':     'Holt-Winters seasonal term',
        'hw_residual':     'Holt-Winters residual',
        'hw_log_forecast': 'Holt-Winters forecast',
    }
    if f in _fixed:
        return _fixed[f]
    if f in _RAW_VAR_LABELS:
        return _RAW_VAR_LABELS[f]
    if f.endswith('_was_missing'):
        base = f[:-len('_was_missing')]
        base_lbl = _RAW_VAR_LABELS.get(base, base.replace('_', ' '))
        return f'Imputed reading ({base_lbl.split(" (")[0].lower()})'
    m = _re.search(r'_lag_(\d+)$', f)
    if m:
        return f'Load {_dur_label(int(m.group(1)))} earlier'
    m = _re.search(r'_roll_(mean|std|min|max)_(\d+)$', f)
    if m:
        stat = {'mean': 'Mean load', 'std': 'Standard deviation',
                'min': 'Minimum load', 'max': 'Maximum load'}[m.group(1)]
        return f'{stat} ({_dur_label(int(m.group(2)))})'
    m = _re.search(r'_ewm_(\d+)$', f)
    if m:
        return f'Smoothed load ({_dur_label(int(m.group(1)))})'
    m = _re.search(r'^f(?:ourier_)?([dw])_(sin|cos)_(\d+)$', f)
    if m:
        scope = 'Daily' if m.group(1) == 'd' else 'Weekly'
        trig = 'sine' if m.group(2) == 'sin' else 'cosine'
        return f'{scope} harmonic {m.group(3)} ({trig})'
    m = _re.search(r'hw_res(?:idual)?_lag_?(\d+)$', f)
    if m:
        return f'Holt-Winters residual ({_dur_label(int(m.group(1)))} earlier)'
    return f.replace('_', ' ')

print(f"Environment configured. Random seed: {RANDOM_STATE}")


Environment configured. Random seed: 42
